# Feature Engineering - Analisis Perilaku Keuangan Nasabah

## 1. Load Library & Data

In [1]:
import pandas as pd
import numpy as np

df_trx = pd.read_csv('/Users/anargyaisadhimaheswara/Documents/DBS Coding Camp/CAPSTONE/DATA-SCIENCE/data-ds/data_transaksi.csv')
df_nas = pd.read_csv('/Users/anargyaisadhimaheswara/Documents/DBS Coding Camp/CAPSTONE/DATA-SCIENCE/data-ds/data_nasabah.csv')

print('data_transaksi:', df_trx.shape)
print('data_nasabah  :', df_nas.shape)

data_transaksi: (53146, 23)
data_nasabah  : (500, 11)


## 2. Preprocessing Awal

In [2]:
df_trx['timestamp'] = pd.to_datetime(df_trx['timestamp'])

df_trx['bulan']      = df_trx['timestamp'].dt.month
df_trx['hari_minggu'] = df_trx['timestamp'].dt.dayofweek
df_trx['jam']        = df_trx['timestamp'].dt.hour
df_trx['hari_bulan'] = df_trx['timestamp'].dt.day

df_trx = df_trx.sort_values(['id_user', 'timestamp']).reset_index(drop=True)
print('Timestamp dikonversi dan data diurutkan.')

Timestamp dikonversi dan data diurutkan.


## 3. Feature Engineering per User per Bulan

In [3]:
df_debit = (
    df_trx[df_trx['tipe_mutasi'] == 'Debit']
    .merge(df_nas[['id_user', 'gaji_bulanan']], on='id_user', how='left')
    .copy()
)

all_features = []

for uid in df_debit['id_user'].unique():
    u_debit   = df_debit[df_debit['id_user'] == uid]
    u_all     = df_trx[df_trx['id_user'] == uid]
    gaji      = u_debit['gaji_bulanan'].iloc[0]
    prev_bal  = None

    for bulan in sorted(u_debit['bulan'].unique()):
        m_debit = u_debit[u_debit['bulan'] == bulan]
        m_all   = u_all[u_all['bulan'] == bulan]

        if m_debit.empty:
            continue

        total_spend = m_debit['nominal'].sum()
        total_trx   = len(m_debit)

        # --- Core Ratios ---
        wants_nom = m_debit[m_debit['kategori_besar'] == 'Wants']['nominal'].sum()
        needs_nom = m_debit[m_debit['kategori_besar'] == 'Needs']['nominal'].sum()
        wants_ratio        = wants_nom / total_spend if total_spend > 0 else 0
        fixed_costs_ratio  = needs_nom / total_spend if total_spend > 0 else 0

        last_bal   = m_all['sisa_saldo'].iloc[-1]
        net_savings = last_bal if prev_bal is None else last_bal - prev_bal
        savings_rate = net_savings / gaji if gaji > 0 else 0

        # --- Habits & Pain of Paying ---
        wants_count       = (m_debit['kategori_besar'] == 'Wants').sum()
        wants_frequency   = wants_count / total_trx if total_trx > 0 else 0
        small_leaks_ratio = (m_debit['nominal'] < 30_000).sum() / total_trx if total_trx > 0 else 0

        # --- Temporal ---
        night_mask        = (m_debit['jam'] >= 22) | (m_debit['jam'] <= 4)
        night_owl_spending = night_mask.sum() / total_trx if total_trx > 0 else 0

        weekend_mask = m_debit['hari_minggu'] >= 5
        w_nom  = m_debit[weekend_mask]['nominal'].sum()
        wd_nom = m_debit[~weekend_mask]['nominal'].sum()
        w_days  = m_debit[weekend_mask]['timestamp'].dt.date.nunique()
        wd_days = m_debit[~weekend_mask]['timestamp'].dt.date.nunique()
        avg_we  = w_nom / w_days   if w_days  > 0 else 0
        avg_wd  = wd_nom / wd_days if wd_days > 0 else 0
        weekend_surge = avg_we / avg_wd if avg_wd > 0 else (1 if avg_we > 0 else 0)

        early_nom = m_debit[m_debit['hari_bulan'] <= 5]['nominal'].sum()
        early_month_depletion = early_nom / gaji if gaji > 0 else 0

        # --- Survival Indicators ---
        daily_bal = m_all.groupby(m_all['timestamp'].dt.date)['sisa_saldo'].last()
        balance_volatility  = daily_bal.std() / gaji if len(daily_bal) > 1 and gaji > 0 else 0
        survival_mode_days  = (daily_bal < (0.15 * gaji)).sum()

        all_features.append({
            'id_user'              : uid,
            'bulan'                : bulan,
            'wants_ratio'          : wants_ratio,
            'fixed_costs_ratio'    : fixed_costs_ratio,
            'savings_rate'         : savings_rate,
            'wants_frequency'      : wants_frequency,
            'small_leaks_ratio'    : small_leaks_ratio,
            'night_owl_spending'   : night_owl_spending,
            'weekend_surge'        : weekend_surge,
            'early_month_depletion': early_month_depletion,
            'balance_volatility'   : balance_volatility,
            'survival_mode_days'   : survival_mode_days,
        })

        prev_bal = last_bal

df_features = pd.DataFrame(all_features).fillna(0)
print(f'Feature engineering selesai: {len(df_features)} baris ({df_features["id_user"].nunique()} user x bulan)')

Feature engineering selesai: 1500 baris (500 user x bulan)


## 4. Preview & Validasi Hasil

In [4]:
df_features.head(10)

,id_user,bulan,wants_ratio,fixed_costs_ratio,savings_rate,wants_frequency,small_leaks_ratio,night_owl_spending,weekend_surge,early_month_depletion,balance_volatility,survival_mode_days
0,USR-001,1,0.900940,0.099060,0.001871,0.722222,0.333333,0.111111,0.009067,0.995223,0.200906,4
1,USR-001,2,0.170089,0.522909,0.001690,0.333333,0.166667,0.000000,7.774289,0.989353,0.281796,4
2,USR-001,3,0.059316,0.940684,-0.001833,0.142857,0.285714,0.000000,1.000000,0.995125,0.003570,3
3,USR-002,1,0.313902,0.463219,0.012319,0.380952,0.142857,0.047619,0.141356,0.970751,0.227404,4
4,USR-002,2,0.699313,0.300687,-0.009408,0.708333,0.291667,0.041667,2.084027,1.009408,0.327462,1
5,USR-002,3,0.253339,0.311359,0.016386,0.538462,0.076923,0.076923,7.136934,0.983614,0.085477,2
6,USR-003,1,0.650792,0.349208,0.001988,0.520000,0.200000,0.040000,0.089505,0.976738,0.179219,3
7,USR-003,2,0.406701,0.593299,-0.000049,0.633333,0.300000,0.033333,4.324729,0.991945,0.251360,2
8,USR-003,3,0.647317,0.352683,-0.000242,0.648649,0.297297,0.054054,1.809632,0.979344,0.206572,4
9,USR-004,1,0.595869,0.404131,0.013250,0.459459,0.297297,0.000000,0.006686,0.996304,0.115427,6


In [5]:
df_features.describe().round(3)

,bulan,wants_ratio,fixed_costs_ratio,savings_rate,wants_frequency,small_leaks_ratio,night_owl_spending,weekend_surge,early_month_depletion,balance_volatility,survival_mode_days
count,1500.000,1500.000,1500.000,1500.000,1500.000,1500.000,1500.000,1500.000,1500.000,1500.000,1500.000
mean,2.000,0.272,0.625,0.007,0.366,0.208,0.063,9.976,0.851,0.261,5.856
std,0.817,0.185,0.227,0.073,0.176,0.084,0.046,78.718,0.210,0.099,3.063
min,1.000,0.016,0.021,-1.055,0.031,0.000,0.000,0.000,0.066,0.000,0.000
25%,1.000,0.124,0.446,-0.001,0.212,0.149,0.031,0.559,0.776,0.212,4.000
50%,2.000,0.227,0.657,0.002,0.355,0.200,0.059,1.272,0.928,0.279,6.000
75%,3.000,0.391,0.821,0.008,0.500,0.259,0.091,2.842,0.987,0.327,8.000
max,3.000,0.936,0.984,1.059,0.810,0.542,0.250,1934.273,1.635,0.652,18.000


In [6]:
print('Missing values:')
print(df_features.isnull().sum())
print('Range bulan per user:')
print(df_features.groupby('id_user')['bulan'].nunique().describe())

Missing values:
id_user                  0
bulan                    0
wants_ratio              0
fixed_costs_ratio        0
savings_rate             0
wants_frequency          0
small_leaks_ratio        0
night_owl_spending       0
weekend_surge            0
early_month_depletion    0
balance_volatility       0
survival_mode_days       0
dtype: int64
Range bulan per user:
count    500.0
mean       3.0
std        0.0
min        3.0
25%        3.0
50%        3.0
75%        3.0
max        3.0
Name: bulan, dtype: float64


## 5. Simpan Hasil Feature Engineering

In [7]:
out_path = '/Users/anargyaisadhimaheswara/Documents/DBS Coding Camp/CAPSTONE/AI-ENGINEER/data-ai/features_nasabah.csv'
df_features.to_csv(out_path, index=False)
print(f'Tersimpan di: {out_path}')

Tersimpan di: /Users/anargyaisadhimaheswara/Documents/DBS Coding Camp/CAPSTONE/AI-ENGINEER/data-ai/features_nasabah.csv
